# Pemeriksaan dan Validasi Dataset (Kelompok 3)
**Mata Kuliah:** Machine Learning (Kelas C), Bapak Adi Purnawan

**Anggota:** Deliana Br Manalu (2305551036) · Ravi Arnan Irianto (2305551076) · Ezza Putra Wibawa (2305551144) · Devin (2305551173)

---

Tahap ini adalah **pemeriksaan awal dataset**: memastikan data benar dan bersih
sebelum masuk ke preprocessing lanjutan. Belum ada pemodelan di sini.

**Sumber data:** [Stroke Prediction Dataset (Kaggle, fedesoriano)](https://www.kaggle.com/datasets/fedoriano/stroke-prediction-dataset),
5.110 rekam medis pasien.

## 1. Muat Data

Dataset diambil langsung dari URL publik, tidak perlu upload file.

In [ ]:
import pandas as pd
import numpy as np

URL = "https://raw.githubusercontent.com/ray-project/raydp/master/tutorials/dataset/healthcare-dataset-stroke-data.csv"
df = pd.read_csv(URL)

print("Dataset berhasil dimuat.")
print("Jumlah baris :", df.shape[0])
print("Jumlah kolom :", df.shape[1])

## 2. Tampilan 5 Baris Pertama

Lihat bentuk data sekilas: tipe nilai, apakah ada yang sudah berupa angka, dll.

In [ ]:
df.head(5)

## 3. Deskripsi Setiap Kolom

Jelasin arti tiap kolom sebelum kita olah.

In [ ]:
deskripsi = {
    "id": "Nomor identitas unik pasien (tidak dipakai untuk prediksi)",
    "gender": "Jenis kelamin: Male, Female, Other",
    "age": "Usia pasien dalam tahun (numerik, float)",
    "hypertension": "Riwayat hipertensi: 0 = tidak, 1 = ya (sudah berupa angka biner)",
    "heart_disease": "Riwayat penyakit jantung: 0 = tidak, 1 = ya (sudah berupa angka biner)",
    "ever_married": "Pernah menikah: Yes / No",
    "work_type": "Tipe pekerjaan: Private, Self-employed, Govt_job, children, Never_worked",
    "Residence_type": "Tipe tempat tinggal: Urban / Rural",
    "avg_glucose_level": "Kadar glukosa rata-rata dalam darah (numerik, float)",
    "bmi": "Indeks massa tubuh (numerik, float)",
    "smoking_status": "Status merokok: formerly smoked, never smoked, smokes, Unknown",
    "stroke": "Target/ label: 0 = tidak stroke, 1 = stroke (sudah berupa angka biner)",
}

for kolom, arti in deskripsi.items():
    print(f"- {kolom}: {arti}")

## 4. Struktur dan Tipe Data

Cek jumlah kolom, tipe data masing-masing, dan apakah ada nilai kosong secara otomatis.

In [ ]:
df.info()

## 5. Cek Data Kosong (Missing Values) dengan Perulangan

Gunakan fungsi perulangan untuk mengecek kolom mana yang punya nilai kosong, berapa jumlahnya, dan persentasenya.

In [ ]:
# Fungsi untuk cek missing value pakai perulangan
def cek_kosong(dataframe):
    print("=" * 55)
    print("PEMERIKSAAN DATA KOSONG")
    print("=" * 55)
    
    total_baris = len(dataframe)
    ada_kosong = False
    
    for kolom in dataframe.columns:
        jumlah_kosong = dataframe[kolom].isna().sum()
        if jumlah_kosong > 0:
            persen = (jumlah_kosong / total_baris) * 100
            print(f"  {kolom}: {jumlah_kosong} baris kosong ({persen:.2f}%)")
            ada_kosong = True
    
    if not ada_kosong:
        print("  Tidak ada data kosong.")
    
    print("=" * 55)

cek_kosong(df)

### 5a. Cek Missing Value Tersembunyi (Non-NaN)

Tidak semua data kosong berupa NaN. Kadang ia menyamar sebagai "Unknown", "N/A", dll. Ini perlu dicek manual dengan perulangan juga.

In [ ]:
# Cek nilai tersembunyi-kosong pada kolom kategorikal
def cek_tersembunyi(dataframe):
    print("=" * 55)
    print("PEMERIKSAAN NILAI KOSONG TERSEMBUNYI")
    print("=" * 55)
    
    mencurigakan = ["Unknown", "unknown", "N/A", "NA", "?", "-", ""]
    total_baris = len(dataframe)
    ditemukan = False
    
    for kolom in dataframe.columns:
        if pd.api.types.is_string_dtype(dataframe[kolom]):
            for nilai in mencurigakan:
                jumlah = (dataframe[kolom] == nilai).sum()
                if jumlah > 0:
                    persen = (jumlah / total_baris) * 100
                    print(f"  {kolom}: {jumlah} baris ({persen:.2f}%) bernilai '{nilai}'")
                    ditemukan = True
    
    if not ditemukan:
        print("  Tidak ada nilai tersembunyi-kosong.")
    
    print("=" * 55)

cek_tersembunyi(df)

## 6. Deskripsi Statistik Dasar

Cek rentang nilai, mean, standar deviasi tiap kolom numerik. Dari sini kita lihat apakah ada nilai yang tidak masuk akal (outlier ekstrem).

In [ ]:
df.describe().T.round(2)

## 7. Cek Duplikat Data

Pastikan tidak ada baris yang duplikat.

In [ ]:
print("Baris duplikat:", df.duplicated().sum())
if "id" in df.columns:
    print("ID duplikat:", df.id.duplicated().sum())
    print("ID unik:", df.id.nunique(), "dari", len(df), "baris")

## 8. Eksplorasi Distribusi Kategorikal

Lihat sebaran nilai tiap kolom kategorikal untuk memastikan tidak ada kategori aneh.

In [ ]:
for kolom in df.columns[df.dtypes.apply(lambda t: pd.api.types.is_string_dtype(t))]:
    print(f"--- {kolom} ---")
    print(df[kolom].value_counts().to_string())
    print()

## 9. Distribusi Kelas Target (Stroke)

Cek apakah data seimbang atau tidak antara pasien stroke dan tidak stroke.

In [ ]:
jumlah_stroke = df.stroke.value_counts()
persen_stroke = (jumlah_stroke / len(df) * 100).round(2)

print("Distribusi kelas target (stroke):")
print(f"  0 (Tidak stroke): {jumlah_stroke[0]} ({persen_stroke[0]}%)")
print(f"  1 (Stroke):       {jumlah_stroke[1]} ({persen_stroke[1]}%)")
print(f"  Rasio: {jumlah_stroke[0]/jumlah_stroke[1]:.1f} : 1")

## 10. Perbaikan Data Kosong

Dari pengecekan di atas, kita tahu:
- `bmi`: 201 baris kosong (NaN)
- `smoking_status`: 1.544 baris bernilai "Unknown" (tersembunyi-kosong)

Kita akan isi sementara dulu agar dataset siap diproses. Untuk `bmi` diisi median, untuk `smoking_status` "Unknown" kita biarkan sebagai kategori sendiri (tidak dihapus karena 30% terlalu besar).

In [ ]:
print("Sebelum perbaikan:")
print(f"  bmi kosong: {df.bmi.isna().sum()}")
print(f"  smoking_status Unknown: {(df.smoking_status == 'Unknown').sum()}")

# Isi bmi kosong dengan median
df_bersih = df.copy()
df_bersih.bmi = df_bersih.bmi.fillna(df_bersih.bmi.median())

print()
print("Setelah perbaikan:")
print(f"  bmi kosong: {df_bersih.bmi.isna().sum()}")
print(f"  Total data kosong tersisa: {df_bersih.isna().sum().sum()}")

## 11. Verifikasi Final: Dataset Sudah Bersih

Konfirmasi tidak ada data kosong tersisa dan ukuran dataset masih utuh.

In [ ]:
# Verifikasi final pakai perulangan
def verifikasi_bersih(dataframe):
    print("=" * 55)
    print("VERIFIKASI FINAL")
    print("=" * 55)
    
    total_kosong = 0
    for kolom in dataframe.columns:
        jk = dataframe[kolom].isna().sum()
        total_kosong += jk
        if jk > 0:
            print(f"  [!] {kolom}: masih {jk} kosong")
    
    if total_kosong == 0:
        print("  Semua kolom bersih. Tidak ada data kosong.")
    
    print(f"  Ukuran dataset: {dataframe.shape}")
    print("=" * 55)

verifikasi_bersih(df_bersih)

## 12. Pemeriksaan Nilai Tidak Masuk Akal (Outlier)

Selain nilai kosong, perlu dicek apakah ada nilai yang secara medis tidak masuk akal. 
Nilai ekstrem belum tentu salah, tapi wajib diperiksa sebelum diproses.


In [ ]:
print("Rentang nilai tiap kolom numerik:")
for kolom in df.select_dtypes(include="number").columns:
    print(f"  {kolom}: {df[kolom].min()} - {df[kolom].max()}")

print()
print("Pengecekan kasus mencurigakan:")

# BMI: nilai di atas 60 sangat jarang secara medis
bmi_timgi = (df.bmi > 60).sum()
print(f"  BMI > 60: {bmi_timgi} baris")

# Usia: balita (di bawah 2 tahun) janggal ada di dataset stroke
usia_balita = (df.age < 2).sum()
print(f"  Usia < 2 tahun: {usia_balita} baris")

if "gender" in df.columns:
    other = (df.gender == "Other").sum()
    print(f"  gender = 'Other': {other} baris")


In [ ]:
# Lihat baris yang ber-BMI sangat tinggi untuk diperiksa
if "bmi" in df.columns:
    df[df.bmi > 60][["age", "bmi", "avg_glucose_level", "stroke"]]


## 13. Simpan Dataset Bersih

Setelah diyakini benar dan bersih, simpan hasilnya ke file CSV. 
Notebook berikutnya cukup membaca file ini, tidak perlu mengulang pengecekan.


In [ ]:
import pathlib

folder = pathlib.Path("../artifacts")
folder.mkdir(exist_ok=True)
jalan = folder / "stroke_pemeriksaan_awal.csv"
df_bersih.to_csv(jalan, index=False)

print(f"Dataset bersih tersimpan di: {jalan}")
print(f"Ukuran: {df_bersih.shape}")


## Ringkasan

Dataset sudah diverifikasi:
- 5.110 baris x 12 kolom, tidak ada duplikat
- Missing value `bmi` (201 baris) sudah diisi median
- `smoking_status` "Unknown" (1.544 baris) dibiarkan sebagai kategori
- Tidak ada data kosong tersisa
- Kelas target tidak seimbang (stroke hanya 4,87%) -- akan ditangani saat pemodelan

Dataset siap untuk tahap preprocessing lanjutan dan pemilihan fitur.
Minggu depan: eksplorasi fitur mana yang paling berpengaruh terhadap stroke, lalu implementasi model.